## 1. Thư viện và đường dẫn

In [1]:
import json
from pathlib import Path
import sys

import joblib
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

project_dir = Path.cwd().resolve()
if project_dir.name == "softmax":
    project_dir = project_dir.parents[1]
elif project_dir.name == "models":
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from models.common import read_split, timestamp, write_json
from models.reporting import SUMMARY_COLUMNS

model_dir = project_dir / "models" / "softmax"
pd.set_option("display.max_columns", None)

## 2. Đọc model và dữ liệu

In [2]:
artifact = joblib.load(model_dir / "training_results.joblib")
search = artifact["search"]
model = joblib.load(model_dir / "model.joblib")
X_train, y_train = read_split("train")
X_test, y_test = read_split("test")
classes = model.classes_.tolist()
results = search.cv_results_
best_index = int(search.best_index_)

## 3. Siêu tham số tốt nhất

In [3]:
display(pd.DataFrame([search.best_params_]))

,model__C
0,100.0


## 4. Bảng tổng hợp

In [4]:
train_prediction = model.predict(X_train)
prediction = model.predict(X_test)

options = {"labels": classes, "average": "macro", "zero_division": 0}
summary = pd.DataFrame([{
    "Model": "Softmax", "Configuration": "GridSearch",
    "CV_accuracy_mean": float(results["mean_test_accuracy"][best_index]),
    "CV_f1_macro_mean": float(results["mean_test_f1_macro"][best_index]),
    "CV_f1_macro_std": float(results["std_test_f1_macro"][best_index]),
    "Test_accuracy": float(accuracy_score(y_test, prediction)),
    "Test_precision_macro": float(precision_score(y_test, prediction, **options)),
    "Test_recall_macro": float(recall_score(y_test, prediction, **options)),
    "Test_f1_macro": float(f1_score(y_test, prediction, **options)),
    "Train_f1_macro": float(f1_score(y_train, train_prediction, **options)),
    "CV_train_validation_gap_F1": float(
        results["mean_train_f1_macro"][best_index] - results["mean_test_f1_macro"][best_index]
    ),
    "Preprocessing": "StandardScaler",
    "Best_grid_parameters": json.dumps(search.best_params_, sort_keys=True),
}], columns=SUMMARY_COLUMNS)
display(summary)

,Model,Configuration,CV_accuracy_mean,CV_f1_macro_mean,CV_f1_macro_std,Test_accuracy,Test_precision_macro,Test_recall_macro,Test_f1_macro,Train_f1_macro,CV_train_validation_gap_F1,Preprocessing,Best_grid_parameters
0,Softmax,GridSearch,0.977841,0.977581,0.005849,0.984091,0.985197,0.984091,0.984049,0.990316,0.0126,StandardScaler,"{""model__C"": 100.0}"


## 5. Kết quả GridSearch

In [5]:
grid_rows = []
for i, parameters in enumerate(results["params"]):
    grid_rows.append({
        "Model": "Softmax", "Candidate": i + 1,
        "Parameters": json.dumps(parameters, sort_keys=True),
        "Rank_CV_F1_macro": int(results["rank_test_f1_macro"][i]),
        "Selected": i == best_index,
        "CV_accuracy_mean": float(results["mean_test_accuracy"][i]),
        "CV_f1_macro_mean": float(results["mean_test_f1_macro"][i]),
        "CV_f1_macro_std": float(results["std_test_f1_macro"][i]),
    })
grid = pd.DataFrame(grid_rows)
display(grid)

,Model,Candidate,Parameters,Rank_CV_F1_macro,Selected,CV_accuracy_mean,CV_f1_macro_mean,CV_f1_macro_std
0,Softmax,1,"{""model__C"": 0.001}",6,False,0.668750,0.615725,0.016439
1,Softmax,2,"{""model__C"": 0.01}",5,False,0.790909,0.768258,0.015654
2,Softmax,3,"{""model__C"": 0.1}",4,False,0.931818,0.930149,0.009726
3,Softmax,4,"{""model__C"": 1.0}",3,False,0.968182,0.967824,0.006765
4,Softmax,5,"{""model__C"": 10.0}",2,False,0.977273,0.976997,0.007324
5,Softmax,6,"{""model__C"": 100.0}",1,True,0.977841,0.977581,0.005849


## 6. Lưu kết quả đánh giá

In [6]:
evaluated_at = timestamp()
evaluation = {
    "model_name": "Softmax", "evaluated_at": evaluated_at,
    "tables": {
        "Tong_hop": summary.to_dict(orient="records"),
        "GridSearch": grid.to_dict(orient="records"),
    },
}
selected = summary.iloc[0]
metadata = {
    "model_file": "model.joblib", "model_name": "Softmax",
    "estimator": "LogisticRegression", "configuration": "GridSearch",
    "parameters": search.best_params_, "preprocessing": "StandardScaler",
    "input_format": "Unscaled feature values in the order listed in features",
    "features": X_train.columns.tolist(), "classes": classes,
    "selection_metric": "CV_f1_macro_mean",
    "cv_f1_macro_mean": float(selected["CV_f1_macro_mean"]),
    "cv_f1_macro_std": float(selected["CV_f1_macro_std"]),
    "train_samples": len(X_train), "test_samples": len(X_test),
    "n_splits": search.n_splits_, "cv_method": str(search.cv),
    "training_finished_at": artifact["finished_at"],
    "versions": artifact["versions"], "evaluated_at": evaluated_at,
    "test_metrics": {
        "accuracy": float(selected["Test_accuracy"]),
        "precision_macro": float(selected["Test_precision_macro"]),
        "recall_macro": float(selected["Test_recall_macro"]),
        "f1_macro": float(selected["Test_f1_macro"]),
    },
}
write_json(model_dir / "model_metadata.json", metadata)
write_json(model_dir / "evaluation_results.json", evaluation)